In [ ]:
import FinanceDataReader as fdr
import pandas as pd
import numpy as np
import requests


In [18]:
etf_list = fdr.StockListing("ETF/KR")
etf_list.head()

c:\Users\jhlee2026\Documents\이종희_Local\code\stock\etf_predict\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'finance.naver.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,Symbol,Category,Name,Price,RiseFall,Change,ChangeRate,NAV,EarningRate,Volume,Amount,MarCap
0,069500,1,KODEX 200,115500,2,2230,1.97,115584.0,51.3036,24424226,2782742,249884
1,360750,4,TIGER 미국S&P500,26545,2,260,0.99,26551.0,6.4452,28935198,765872,168906
2,396500,2,TIGER 반도체TOP10,45920,2,450,0.99,45904.0,62.2290,27079617,1223917,122009
3,102110,1,TIGER 200,115490,2,2320,2.05,115593.0,51.2879,8824777,1004827,100996
4,133690,4,TIGER 미국나스닥100,184070,2,2235,1.23,184170.0,14.2075,1576248,289329,95569


# 1, 2번


In [46]:
# =========================================================
# 1. ETF 데이터 수집
# =========================================================

def load_etf_price(
    etf_code,
    start_date,
    end_date=None,
    verify_ssl=False
):
    """
    FinanceDataReader 데이터 수집 함수

    회사망 SSL 이슈 대응:
    - requests.Session() 생성
    - session.verify 설정
    - FinanceDataReader 내부 requests.get을 일시적으로 patch
    - 수집 후 원래 requests.get으로 복구
    """

    session = requests.Session()
    session.verify = verify_ssl
    session.headers.update({
        "User-Agent": "Mozilla/5.0"
    })

    original_get = requests.get

    def custom_get(*args, **kwargs):
        kwargs["verify"] = verify_ssl
        return session.get(*args, **kwargs)

    # FinanceDataReader 내부 requests.get 일시 덮어쓰기
    requests.get = custom_get

    df = fdr.DataReader(etf_code, start_date, end_date)


    df = df.reset_index().rename(columns={"index": "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    return df


# =========================================================
# 2. 타겟 생성
#    N일 뒤 수익률이 threshold 이상이면 1, 아니면 0
# =========================================================

def make_target(df, price_col="Close", n_days=5, threshold=0.01):
    df = df.copy()

    future_close_col = f"future_close_{n_days}d"
    future_return_col = f"future_return_{n_days}d"
    target_col = f"target_{n_days}d_up"

    df[future_close_col] = df[price_col].shift(-n_days)

    df[future_return_col] = df[future_close_col] / df[price_col] - 1

    df[target_col] = np.where(
        df[future_return_col] >= threshold,
        1,
        0
    )

    # 미래값 없는 마지막 N일 제거
    df = df.dropna(subset=[future_return_col]).copy()

    return df, target_col


# =========================================================
# 3. ETF 자체 파생변수 생성
# =========================================================

def make_price_features(df):
    df = df.copy()

    # -----------------------------
    # 수익률
    # -----------------------------
    df["return_1d"] = df["Close"].pct_change(1)
    df["return_3d"] = df["Close"].pct_change(3)
    df["return_5d"] = df["Close"].pct_change(5)
    df["return_10d"] = df["Close"].pct_change(10)
    df["return_20d"] = df["Close"].pct_change(20)

    # -----------------------------
    # 이동평균
    # -----------------------------
    df["ma_5"] = df["Close"].rolling(5).mean()
    df["ma_10"] = df["Close"].rolling(10).mean()
    df["ma_20"] = df["Close"].rolling(20).mean()
    df["ma_60"] = df["Close"].rolling(60).mean()

    # -----------------------------
    # 종가 / 이동평균 비율
    # -----------------------------
    df["close_ma5_ratio"] = df["Close"] / df["ma_5"] - 1
    df["close_ma10_ratio"] = df["Close"] / df["ma_10"] - 1
    df["close_ma20_ratio"] = df["Close"] / df["ma_20"] - 1
    df["close_ma60_ratio"] = df["Close"] / df["ma_60"] - 1

    # -----------------------------
    # 변동성
    # -----------------------------
    df["volatility_5d"] = df["return_1d"].rolling(5).std()
    df["volatility_10d"] = df["return_1d"].rolling(10).std()
    df["volatility_20d"] = df["return_1d"].rolling(20).std()

    # -----------------------------
    # 거래량 변화율
    # -----------------------------
    df["volume_change_1d"] = df["Volume"].pct_change(1)
    df["volume_change_5d"] = df["Volume"].pct_change(5)
    df["volume_change_20d"] = df["Volume"].pct_change(20)

    # -----------------------------
    # 거래량 이동평균 비율
    # -----------------------------
    df["volume_ma5"] = df["Volume"].rolling(5).mean()
    df["volume_ma20"] = df["Volume"].rolling(20).mean()

    df["volume_ma5_ratio"] = df["Volume"] / df["volume_ma5"] - 1
    df["volume_ma20_ratio"] = df["Volume"] / df["volume_ma20"] - 1

    # -----------------------------
    # 일중 가격 움직임
    # -----------------------------
    df["high_low_range"] = (df["High"] - df["Low"]) / df["Close"]
    df["open_close_return"] = df["Close"] / df["Open"] - 1

    # -----------------------------
    # 캔들 위치
    # 종가가 당일 고저 범위에서 어디쯤 있는지
    # -----------------------------
    df["close_position"] = (
        (df["Close"] - df["Low"]) / (df["High"] - df["Low"])
    )

    df["close_position"] = df["close_position"].replace([np.inf, -np.inf], np.nan)

    return df


# =========================================================
# 4. 모델용 데이터셋 생성
# =========================================================

def make_model_dataset(
    etf_code,
    start_date,
    end_date=None,
    n_days=5,
    threshold=0.01
):
    # 1. 가격 데이터 수집
    df = load_etf_price(
        etf_code=etf_code,
        start_date=start_date,
        end_date=end_date
    )

    # 2. 타겟 생성
    df, target_col = make_target(
        df=df,
        price_col="Adj Close",
        n_days=n_days,
        threshold=threshold
    )

    # 3. 파생변수 생성
    df = make_price_features(df)

    # 4. 미래값 컬럼명
    future_close_col = f"future_close_{n_days}d"
    future_return_col = f"future_return_{n_days}d"

    # 5. 모델에 제외할 컬럼
    exclude_cols = [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "Change",
        future_close_col,
        future_return_col,
        target_col
    ]

    # 실제 존재하는 컬럼만 제외
    exclude_cols = [col for col in exclude_cols if col in df.columns]

    # 6. feature 컬럼
    feature_cols = [
        col for col in df.columns
        if col not in exclude_cols
    ]

    # 7. 결측 / inf 제거
    df = df.replace([np.inf, -np.inf], np.nan)

    final_df = df.dropna(subset=feature_cols + [target_col]).copy()

    X = final_df[feature_cols].copy()
    y = final_df[target_col].copy()

    return final_df, X, y, feature_cols, target_col




In [48]:
# =========================================================
# 설정값
# =========================================================

ETF_CODE = "SMH"
START_DATE = "2020-01-01"
END_DATE = None            # None이면 오늘까지

N_DAYS = 5                 # N일 뒤
THRESHOLD = 0.05           # N일 뒤 N% 이상 상승하면 1


# =========================================================
# 실행
# =========================================================

final_df, X, y, feature_cols, target_col = make_model_dataset(
    etf_code=ETF_CODE,
    start_date=START_DATE,
    end_date=END_DATE,
    n_days=N_DAYS,
    threshold=THRESHOLD
)

final_df = final_df[['Date'] + [target_col] + feature_cols]

print("ETF_CODE:", ETF_CODE)
print("N_DAYS:", N_DAYS)
print("THRESHOLD:", THRESHOLD)
print("-" * 50)

print("final_df shape:", final_df.shape)
print("-" * 50)

print("target_col:", target_col)
print("target 비율:")
print(y.value_counts(normalize=True))
print("-" * 50)

final_df

c:\Users\jhlee2026\Documents\이종희_Local\code\stock\etf_predict\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'query2.finance.yahoo.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


ETF_CODE: SMH
N_DAYS: 5
THRESHOLD: 0.05
--------------------------------------------------
final_df shape: (1531, 29)
--------------------------------------------------
target_col: target_5d_up
target 비율:
target_5d_up
0    0.835402
1    0.164598
Name: proportion, dtype: float64
--------------------------------------------------


,Date,target_5d_up,Adj Close,return_1d,return_3d,return_5d,return_10d,return_20d,ma_5,ma_10,...,volume_change_1d,volume_change_5d,volume_change_20d,volume_ma5,volume_ma20,volume_ma5_ratio,volume_ma20_ratio,high_low_range,open_close_return,close_position
59,2020-03-26,0,58.641945,0.060648,0.161340,0.183356,0.120707,-0.059496,55.798000,54.662500,...,-0.060072,-0.104876,-0.645148,12305920.0,13522310.0,-0.251352,-0.318696,0.047650,0.042123,0.951724
60,2020-03-27,0,55.399582,-0.055291,-0.005535,0.137839,-0.034833,-0.128666,57.191000,54.455000,...,0.364558,-0.084585,-0.348558,12073600.0,13185990.0,0.041230,-0.046609,0.035829,-0.021612,0.009709
61,2020-03-30,0,57.572399,0.039221,0.041304,0.140158,0.171913,-0.124799,58.660000,55.331500,...,-0.272507,-0.304707,-0.350593,11272000.0,12939120.0,-0.188644,-0.293182,0.034059,0.020321,0.899262
62,2020-03-31,0,56.435406,-0.019749,-0.037627,0.013059,0.062687,-0.112979,58.811000,55.677000,...,0.117871,-0.345840,-0.309263,10191000.0,12710250.0,0.003199,-0.195641,0.042513,-0.021714,0.156626
63,2020-04-01,1,53.636284,-0.049599,-0.031829,-0.029888,0.101514,-0.198084,58.468000,56.190000,...,0.265308,0.319785,0.197467,10817880.0,12816910.0,0.195798,0.009292,0.053984,-0.018600,0.191348
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1585,2026-04-23,1,481.850006,0.010528,0.038559,0.059477,0.119774,0.207584,470.291998,459.147998,...,0.355383,0.646286,0.437289,7034640.0,8725450.0,0.509629,0.217095,0.026751,0.003018,0.516681
1586,2026-04-24,0,506.440002,0.051032,0.089915,0.091089,0.159220,0.329797,478.747998,466.103998,...,0.187821,0.755741,-0.148039,8120580.0,8615855.0,0.553374,0.464080,0.027901,0.013874,0.777071
1587,2026-04-27,0,506.260010,-0.000355,0.061720,0.091172,0.141923,0.352732,487.208002,472.395999,...,-0.283337,1.297324,-0.234621,9141600.0,8477295.0,-0.011092,0.066401,0.024395,-0.005754,0.689069
1588,2026-04-28,1,491.209991,-0.029728,0.019425,0.057139,0.086748,0.354950,492.517999,476.316998,...,0.400942,1.262137,0.038252,10554840.0,8500625.0,0.199904,0.489867,0.027218,0.006186,0.592370


# 3,4 번 (3번은 일단 생략)


In [63]:
external_tickers = {
    "QQQ": "NASDAQ100_ETF",
    "SPY": "SP500_ETF",
    "SOXX": "SEMICONDUCTOR_ETF",
    "DIA": "DOW_ETF",
    "GLD": "GOLD_ETF",
    "USO": "OIL_ETF",
}

extra_candidates = {
    "VIX": "MARKET_VOLATILITY",
    "DXY": "DOLLAR_INDEX",
    "US10Y": "US_10Y_YIELD",
}
# =========================================================
# 외부 X지표 파생변수 별도 수집
# =========================================================

EXTERNAL_TICKERS = {
    "QQQ": "qqq",
    "SPY": "spy",
    "SOXX": "soxx",
    "GLD": "gold",
    "USO": "oil",
}


def get_price_col(df):
    if "Adj Close" in df.columns:
        return "Adj Close"
    return "Close"


def make_external_price_features(df, prefix):
    """
    외부 X지표 1개에 대한 파생변수 생성
    Adj Close 기준
    """
    df = df.copy()

    price_col = get_price_col(df)

    # Date 정리
    if "Date" not in df.columns:
        df = df.reset_index()

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    # 사용할 원천 컬럼
    use_cols = ["Date", price_col]

    if "Volume" in df.columns:
        use_cols.append("Volume")

    temp = df[use_cols].copy()

    # 기준 가격 컬럼명 통일
    temp = temp.rename(columns={
        price_col: f"{prefix}_adj_close"
    })

    close_col = f"{prefix}_adj_close"

    # -----------------------------
    # 수익률
    # -----------------------------
    temp[f"{prefix}_return_1d"] = temp[close_col].pct_change(1)
    temp[f"{prefix}_return_3d"] = temp[close_col].pct_change(3)
    temp[f"{prefix}_return_5d"] = temp[close_col].pct_change(5)
    temp[f"{prefix}_return_10d"] = temp[close_col].pct_change(10)
    temp[f"{prefix}_return_20d"] = temp[close_col].pct_change(20)

    # -----------------------------
    # 이동평균
    # -----------------------------
    temp[f"{prefix}_ma_5"] = temp[close_col].rolling(5).mean()
    temp[f"{prefix}_ma_10"] = temp[close_col].rolling(10).mean()
    temp[f"{prefix}_ma_20"] = temp[close_col].rolling(20).mean()
    temp[f"{prefix}_ma_60"] = temp[close_col].rolling(60).mean()

    # -----------------------------
    # 가격 / 이동평균 비율
    # -----------------------------
    temp[f"{prefix}_close_ma5_ratio"] = temp[close_col] / temp[f"{prefix}_ma_5"] - 1
    temp[f"{prefix}_close_ma10_ratio"] = temp[close_col] / temp[f"{prefix}_ma_10"] - 1
    temp[f"{prefix}_close_ma20_ratio"] = temp[close_col] / temp[f"{prefix}_ma_20"] - 1
    temp[f"{prefix}_close_ma60_ratio"] = temp[close_col] / temp[f"{prefix}_ma_60"] - 1

    # -----------------------------
    # 변동성
    # -----------------------------
    temp[f"{prefix}_volatility_5d"] = temp[f"{prefix}_return_1d"].rolling(5).std()
    temp[f"{prefix}_volatility_10d"] = temp[f"{prefix}_return_1d"].rolling(10).std()
    temp[f"{prefix}_volatility_20d"] = temp[f"{prefix}_return_1d"].rolling(20).std()

    # -----------------------------
    # 거래량 파생변수
    # -----------------------------
    if "Volume" in temp.columns:
        temp = temp.rename(columns={
            "Volume": f"{prefix}_volume"
        })

        volume_col = f"{prefix}_volume"

        temp[f"{prefix}_volume_change_1d"] = temp[volume_col].pct_change(1)
        temp[f"{prefix}_volume_change_5d"] = temp[volume_col].pct_change(5)
        temp[f"{prefix}_volume_change_20d"] = temp[volume_col].pct_change(20)

        temp[f"{prefix}_volume_ma5"] = temp[volume_col].rolling(5).mean()
        temp[f"{prefix}_volume_ma20"] = temp[volume_col].rolling(20).mean()

        temp[f"{prefix}_volume_ma5_ratio"] = temp[volume_col] / temp[f"{prefix}_volume_ma5"] - 1
        temp[f"{prefix}_volume_ma20_ratio"] = temp[volume_col] / temp[f"{prefix}_volume_ma20"] - 1

        # 원본 거래량 제거
        temp = temp.drop(columns=[volume_col])

    # -----------------------------
    # 원본 가격 제거
    # 나중에 모델에는 가격 레벨보다 파생변수를 쓰기 위해 제거
    # 필요하면 이 줄 주석 처리
    # -----------------------------
    temp = temp.drop(columns=[close_col])

    # inf 처리
    temp = temp.replace([np.inf, -np.inf], np.nan)

    return temp


def load_single_external_features(
    ticker,
    prefix,
    start_date,
    end_date=None,
    verify_ssl=False
):
    """
    외부 X지표 1개 수집 + 파생변수 생성
    """
    raw_df = load_etf_price(
        etf_code=ticker,
        start_date=start_date,
        end_date=end_date,
        verify_ssl=verify_ssl
    )

    feature_df = make_external_price_features(
        df=raw_df,
        prefix=prefix
    )

    print(f"{ticker} -> {prefix}, shape: {feature_df.shape}")

    return feature_df


def load_external_features(
    external_tickers,
    start_date,
    end_date=None,
    verify_ssl=False
):
    """
    여러 외부 X지표 수집 + 파생변수 생성 + Date 기준 병합
    """
    external_df = None

    for ticker, prefix in external_tickers.items():
        print(f"[외부지표 수집] {ticker} -> {prefix}")

        temp = load_single_external_features(
            ticker=ticker,
            prefix=prefix,
            start_date=start_date,
            end_date=end_date,
            verify_ssl=verify_ssl
        )

        if external_df is None:
            external_df = temp.copy()
        else:
            external_df = external_df.merge(
                temp,
                on="Date",
                how="left"
            )

        print("현재 external_df shape:", external_df.shape)

    external_df = external_df.sort_values("Date").reset_index(drop=True)

    return external_df

In [64]:
external_df = load_external_features(
    external_tickers=EXTERNAL_TICKERS,
    start_date=START_DATE,
    end_date=END_DATE,
    verify_ssl=False
)

external_df

[외부지표 수집] QQQ -> qqq


c:\Users\jhlee2026\Documents\이종희_Local\code\stock\etf_predict\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'query2.finance.yahoo.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


QQQ -> qqq, shape: (1595, 24)
현재 external_df shape: (1595, 24)
[외부지표 수집] SPY -> spy


c:\Users\jhlee2026\Documents\이종희_Local\code\stock\etf_predict\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'query2.finance.yahoo.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


SPY -> spy, shape: (1595, 24)
현재 external_df shape: (1595, 47)
[외부지표 수집] SOXX -> soxx


...

c:\Users\jhlee2026\Documents\이종희_Local\code\stock\etf_predict\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'query2.finance.yahoo.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


GLD -> gold, shape: (1595, 24)
현재 external_df shape: (1595, 93)
[외부지표 수집] USO -> oil


c:\Users\jhlee2026\Documents\이종희_Local\code\stock\etf_predict\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'query2.finance.yahoo.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


USO -> oil, shape: (1595, 24)
현재 external_df shape: (1595, 116)


,Date,qqq_return_1d,qqq_return_3d,qqq_return_5d,qqq_return_10d,qqq_return_20d,qqq_ma_5,qqq_ma_10,qqq_ma_20,qqq_ma_60,...,oil_volatility_5d,oil_volatility_10d,oil_volatility_20d,oil_volume_change_1d,oil_volume_change_5d,oil_volume_change_20d,oil_volume_ma5,oil_volume_ma20,oil_volume_ma5_ratio,oil_volume_ma20_ratio
0,2019-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-02,0.016697,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.077878,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-03,-0.009160,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.766273,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-06,0.006443,0.013875,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-0.399273,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-07,-0.000139,-0.002914,NaN,NaN,NaN,207.014587,NaN,NaN,NaN,...,NaN,NaN,NaN,-0.329562,NaN,NaN,2336047.8,NaN,-0.254264,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1590,2026-04-30,0.009326,0.005284,0.025053,0.042578,0.142784,662.993994,656.146991,633.737497,609.984063,...,0.041883,0.046062,0.049034,-0.287802,-0.446652,-0.765626,11742060.0,24366050.0,-0.096062,-0.564390
1591,2026-05-01,0.009600,0.025245,0.015470,0.038992,0.152433,665.047998,658.676996,638.195999,611.136777,...,0.044671,0.036543,0.043435,0.205811,-0.100489,-0.799971,11456100.0,21806800.0,0.117186,-0.413091
1592,2026-05-04,-0.001884,0.017096,0.013023,0.040338,0.143381,666.778003,661.285999,642.414999,612.413474,...,0.045386,0.035867,0.043981,0.185333,1.161177,-0.464230,13086300.0,21149555.0,0.159273,-0.282699
1593,2026-05-05,0.012974,0.020772,0.036590,0.057858,0.158039,671.590002,665.013995,647.065997,613.625602,...,0.047576,0.035729,0.044353,-0.436344,-0.284130,-0.832124,12407520.0,19030285.0,-0.310821,-0.550664


In [75]:
merged_df = final_df.merge(
    external_df,
    on="Date",
    how="left"
)

In [76]:
def remove_high_corr_features(df, target_col=None, exclude_cols=None, corr_threshold=0.95):
    df = df.copy()

    if exclude_cols is None:
        exclude_cols = []

    if target_col is not None:
        exclude_cols = exclude_cols + [target_col]

    exclude_cols = [col for col in exclude_cols if col in df.columns]

    candidate_cols = [
        col for col in df.columns
        if col not in exclude_cols
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    X = df[candidate_cols].replace([np.inf, -np.inf], np.nan).dropna()

    corr_matrix = X.corr().abs()

    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    drop_cols = [
        col for col in upper.columns
        if any(upper[col] > corr_threshold)
    ]

    keep_cols = [col for col in candidate_cols if col not in drop_cols]

    print("기존 변수 수:", len(candidate_cols))
    print("상관계수 제거 변수 수:", len(drop_cols))
    print("남은 변수 수:", len(keep_cols))

    selected_df = df[exclude_cols + keep_cols].copy()

    return selected_df, keep_cols, drop_cols

In [77]:
corr_selected_df, corr_selected_features, corr_drop_cols = remove_high_corr_features(
    df=merged_df,
    target_col=target_col,
    exclude_cols=["Date"],
    corr_threshold=0.95
)

기존 변수 수: 142
상관계수 제거 변수 수: 35
남은 변수 수: 107


In [81]:
corr_selected_df

,Date,target_5d_up,Adj Close,return_1d,return_3d,return_5d,return_10d,return_20d,close_ma5_ratio,close_ma10_ratio,...,oil_volatility_5d,oil_volatility_10d,oil_volatility_20d,oil_volume_change_1d,oil_volume_change_5d,oil_volume_change_20d,oil_volume_ma5,oil_volume_ma20,oil_volume_ma5_ratio,oil_volume_ma20_ratio
0,2020-03-26,0,58.641945,0.060648,0.161340,0.183356,0.120707,-0.059496,0.090720,0.113378,...,0.044073,0.092976,0.091772,0.272068,-0.034641,1.519327,13214385.4,11023797.15,0.188364,0.424509
1,2020-03-27,0,55.399582,-0.055291,-0.005535,0.137839,-0.034833,-0.128666,0.005316,0.055826,...,0.042222,0.088274,0.092303,-0.024928,-0.164812,1.849741,12610062.8,11520742.75,0.214272,0.329085
2,2020-03-30,0,57.572399,0.039221,0.041304,0.140158,0.171913,-0.124799,0.018582,0.079855,...,0.044544,0.082630,0.090335,0.341504,1.269646,2.970565,14908220.2,12289133.35,0.377842,0.671491
3,2020-03-31,0,56.435406,-0.019749,-0.037627,0.013059,0.062687,-0.112979,-0.004098,0.051960,...,0.038830,0.082560,0.090301,-0.238626,0.469947,1.440530,15908222.6,12750697.10,-0.016890,0.226562
4,2020-04-01,1,53.636284,-0.049599,-0.031829,-0.029888,0.101514,-0.198084,-0.047941,-0.009343,...,0.046077,0.066036,0.091689,-0.048651,0.205250,1.568068,16414980.0,13204943.95,-0.093593,0.126748
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1526,2026-04-23,1,481.850006,0.010528,0.038559,0.059477,0.119774,0.207584,0.024576,0.049444,...,0.054884,0.042099,0.048487,0.612644,0.424003,-0.543037,21589160.0,32424090.00,-0.111517,-0.408415
1527,2026-04-24,0,506.440002,0.051032,0.089915,0.091089,0.159220,0.329797,0.057843,0.086539,...,0.030535,0.042122,0.048489,-0.258227,-0.628650,-0.664413,16771780.0,31015580.00,-0.151646,-0.541250
1528,2026-04-27,0,506.260010,-0.000355,0.061720,0.091172,0.141923,0.352732,0.039104,0.071686,...,0.028835,0.041585,0.047011,-0.506649,-0.539544,-0.786273,15126720.0,29724375.00,-0.535947,-0.763844
1529,2026-04-28,1,491.209991,-0.029728,0.019425,0.057139,0.086748,0.354950,-0.002656,0.031267,...,0.023379,0.039806,0.046644,0.701650,-0.487552,-0.704954,12853800.0,28297375.00,-0.070711,-0.577880


In [82]:
# 다중공선성 제거

import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor


def remove_multicollinearity_by_vif(
    df,
    target_col=None,
    exclude_cols=None,
    vif_threshold=10.0,
    min_features=2,
    verbose=True
):
    """
    VIF 기반 다중공선성 제거 함수

    Parameters
    ----------
    df : pd.DataFrame
        후보 변수들이 들어있는 데이터프레임

    target_col : str, optional
        타겟 컬럼명. 있으면 VIF 계산에서 제외

    exclude_cols : list, optional
        VIF 계산에서 제외할 컬럼들. 예: Date, target 등

    vif_threshold : float
        VIF 제거 기준. 보통 10 또는 30 사용

    min_features : int
        최소로 남길 변수 개수

    verbose : bool
        제거 과정 출력 여부

    Returns
    -------
    selected_df : pd.DataFrame
        VIF 제거 후 데이터프레임

    selected_features : list
        최종 선택된 feature 목록

    vif_history : pd.DataFrame
        제거 과정 기록
    """

    df = df.copy()

    if exclude_cols is None:
        exclude_cols = []

    if target_col is not None:
        exclude_cols = exclude_cols + [target_col]

    exclude_cols = list(set([col for col in exclude_cols if col in df.columns]))

    # 숫자형 컬럼만 후보로 사용
    candidate_cols = [
        col for col in df.columns
        if col not in exclude_cols
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    X = df[candidate_cols].copy()

    # inf 제거
    X = X.replace([np.inf, -np.inf], np.nan)

    # 결측 제거
    X = X.dropna().copy()

    # 상수 컬럼 제거
    nunique = X.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()

    if constant_cols:
        if verbose:
            print("[상수 컬럼 제거]", constant_cols)
        X = X.drop(columns=constant_cols)

    selected_features = X.columns.tolist()
    vif_history = []

    while len(selected_features) > min_features:
        X_vif = X[selected_features].copy()

        vif_list = []

        for i, col in enumerate(selected_features):
            try:
                vif_value = variance_inflation_factor(X_vif.values, i)
            except Exception:
                vif_value = np.inf

            vif_list.append({
                "feature": col,
                "vif": vif_value
            })

        vif_df = pd.DataFrame(vif_list).sort_values("vif", ascending=False)

        max_vif = vif_df.iloc[0]["vif"]
        remove_feature = vif_df.iloc[0]["feature"]

        vif_history.append({
            "step": len(vif_history) + 1,
            "removed_feature": remove_feature,
            "max_vif": max_vif,
            "n_features_before": len(selected_features)
        })

        if verbose:
            print(
                f"[VIF 체크] max_vif={max_vif:.2f}, "
                f"feature={remove_feature}, "
                f"n_features={len(selected_features)}"
            )

        if max_vif <= vif_threshold:
            break

        selected_features.remove(remove_feature)

    selected_df = df[exclude_cols + selected_features].copy()

    vif_history = pd.DataFrame(vif_history)

    if verbose:
        print("-" * 50)
        print("최종 선택 변수 개수:", len(selected_features))
        print("최종 선택 변수:")
        print(selected_features)

    return selected_df, selected_features, vif_history

In [83]:
selected_df, selected_features, vif_history = remove_multicollinearity_by_vif(
    df=merged_df,
    target_col=target_col,
    exclude_cols=["Date"],
    vif_threshold=10.0,
    verbose=True
)

[VIF 체크] max_vif=4247015.96, feature=spy_ma_10, n_features=142
[VIF 체크] max_vif=3046211.46, feature=soxx_ma_10, n_features=141
[VIF 체크] max_vif=1421654.94, feature=qqq_ma_10, n_features=140
[VIF 체크] max_vif=1008596.79, feature=spy_ma_20, n_features=139
[VIF 체크] max_vif=626392.16, feature=soxx_ma_20, n_features=138
[VIF 체크] max_vif=286100.94, feature=qqq_ma_20, n_features=137
[VIF 체크] max_vif=227660.88, feature=spy_ma_60, n_features=136
[VIF 체크] max_vif=213945.65, feature=gold_ma_10, n_features=135
[VIF 체크] max_vif=133122.68, feature=ma_5, n_features=134
[VIF 체크] max_vif=66149.08, feature=ma_10, n_features=133
[VIF 체크] max_vif=58186.05, feature=gold_ma_20, n_features=132
[VIF 체크] max_vif=45298.09, feature=qqq_ma_60, n_features=131
[VIF 체크] max_vif=38547.59, feature=oil_ma_10, n_features=130
[VIF 체크] max_vif=35435.67, feature=ma_60, n_features=129
[VIF 체크] max_vif=16319.24, feature=gold_ma_60, n_features=128
[VIF 체크] max_vif=9960.12, feature=oil_ma_5, n_features=127
[VIF 체크] max_vif=7713

In [85]:
selected_df.to_csv("selected_features.csv", index=False)

In [87]:
## 최적 LAG 탐색

import numpy as np
import pandas as pd


# =========================================================
# 1. lag 라벨 함수
# =========================================================

def format_lag_label(lag):
    if pd.isna(lag):
        return "lag_NA"
    lag = int(lag)

    if lag > 0:
        return f"선행 {lag}일"
    elif lag < 0:
        return f"후행 {abs(lag)}일"
    else:
        return "동행"


# =========================================================
# 2. 단일 feature 최적 lag 탐색
#    corr(y_t, x_{t-lag})
#    lag > 0 : X가 target보다 선행
#    lag = 0 : 동행
#    lag < 0 : X가 target보다 후행
# =========================================================

def find_best_lag_for_feature(
    y: pd.Series,
    x: pd.Series,
    max_lag: int = 5,
    allow_negative_lag: bool = False,
    min_obs: int = 30,
):
    rows = []

    y_name = y.name if y.name is not None else "target"
    x_name = x.name if x.name is not None else "feature"

    temp = pd.concat([y, x], axis=1).copy()
    temp.columns = [y_name, x_name]
    temp = temp.replace([np.inf, -np.inf], np.nan).dropna()

    if len(temp) < min_obs:
        return {
            "feature": x_name,
            "best_lag": np.nan,
            "lag_relation": "lag_NA",
            "best_corr": np.nan,
            "abs_best_corr": np.nan,
            "n_obs": len(temp),
            "lag_table": pd.DataFrame()
        }

    if allow_negative_lag:
        lag_range = range(-max_lag, max_lag + 1)
    else:
        # 예측용이므로 현재/과거 X만 허용
        lag_range = range(0, max_lag + 1)

    for lag in lag_range:
        # lag=3이면 y_t와 x_{t-3} 비교
        x_lagged = temp[x_name].shift(lag)

        valid = pd.concat(
            [temp[y_name], x_lagged],
            axis=1
        ).dropna()

        if len(valid) < min_obs:
            corr = np.nan
        else:
            corr = valid.iloc[:, 0].corr(valid.iloc[:, 1])

        rows.append({
            "feature": x_name,
            "lag": lag,
            "lag_relation": format_lag_label(lag),
            "corr": corr,
            "abs_corr": abs(corr) if pd.notna(corr) else np.nan,
            "n_obs": len(valid)
        })

    lag_table = pd.DataFrame(rows)

    if lag_table["abs_corr"].isna().all():
        return {
            "feature": x_name,
            "best_lag": np.nan,
            "lag_relation": "lag_NA",
            "best_corr": np.nan,
            "abs_best_corr": np.nan,
            "n_obs": 0,
            "lag_table": lag_table
        }

    best_row = lag_table.loc[lag_table["abs_corr"].idxmax()]

    return {
        "feature": x_name,
        "best_lag": int(best_row["lag"]),
        "lag_relation": best_row["lag_relation"],
        "best_corr": float(best_row["corr"]),
        "abs_best_corr": float(best_row["abs_corr"]),
        "n_obs": int(best_row["n_obs"]),
        "lag_table": lag_table
    }


# =========================================================
# 3. 전체 feature 최적 lag 탐색
# =========================================================

def find_best_lags_for_all_features(
    df: pd.DataFrame,
    target_col: str,
    date_col: str = "Date",
    max_lag: int = 5,
    allow_negative_lag: bool = False,
    min_obs: int = 30,
):
    df = df.copy()

    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.sort_values(date_col).reset_index(drop=True)

    exclude_cols = [target_col]
    if date_col in df.columns:
        exclude_cols.append(date_col)

    feature_cols = [
        col for col in df.columns
        if col not in exclude_cols
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    y = df[target_col]

    summary_rows = []
    lag_detail_dict = {}

    for col in feature_cols:
        result = find_best_lag_for_feature(
            y=y,
            x=df[col],
            max_lag=max_lag,
            allow_negative_lag=allow_negative_lag,
            min_obs=min_obs
        )

        summary_rows.append({
            "feature": result["feature"],
            "lag_relation": result["lag_relation"],
            "best_lag": result["best_lag"],
            "best_corr": result["best_corr"],
            "abs_best_corr": result["abs_best_corr"],
            "n_obs": result["n_obs"],
        })

        lag_detail_dict[col] = result["lag_table"]

    summary_df = pd.DataFrame(summary_rows)

    summary_df["best_corr"] = summary_df["best_corr"].round(4)
    summary_df["abs_best_corr"] = summary_df["abs_best_corr"].round(4)

    summary_df = (
        summary_df
        .sort_values("abs_best_corr", ascending=False)
        .reset_index(drop=True)
    )

    return summary_df, lag_detail_dict


# =========================================================
# 4. 선택된 lag를 실제 feature table에 적용
# =========================================================

def build_lagged_feature_table(
    df: pd.DataFrame,
    summary_df: pd.DataFrame,
    target_col: str,
    date_col: str = "Date",
    min_abs_corr: float = 0.03,
    max_features: int | None = None,
    keep_original_names: bool = False,
):
    df = df.copy()

    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.sort_values(date_col).reset_index(drop=True)

    required_cols = {"feature", "best_lag", "abs_best_corr"}
    missing_cols = required_cols - set(summary_df.columns)

    if missing_cols:
        raise ValueError(f"summary_df에 필요한 컬럼이 없습니다: {missing_cols}")

    selected_summary = summary_df.copy()

    # 너무 상관 낮은 변수 제거
    selected_summary = selected_summary[
        selected_summary["abs_best_corr"] >= min_abs_corr
    ].copy()

    # lag 없는 변수 제거
    selected_summary = selected_summary[
        selected_summary["best_lag"].notna()
    ].copy()

    # 상위 N개만 선택
    if max_features is not None:
        selected_summary = selected_summary.head(max_features).copy()

    lagged_df = pd.DataFrame()

    if date_col in df.columns:
        lagged_df[date_col] = df[date_col]

    lagged_df[target_col] = df[target_col]

    for _, row in selected_summary.iterrows():
        feature = row["feature"]
        best_lag = int(row["best_lag"])

        if feature not in df.columns:
            continue

        shifted = df[feature].shift(best_lag)

        if keep_original_names:
            new_col = feature
        else:
            new_col = f"{feature}_lag{best_lag}"

        lagged_df[new_col] = shifted

    lagged_df = lagged_df.replace([np.inf, -np.inf], np.nan)
    lagged_df = lagged_df.dropna().reset_index(drop=True)

    X_lagged = lagged_df.drop(columns=[target_col])

    if date_col in X_lagged.columns:
        X_lagged = X_lagged.drop(columns=[date_col])

    y_lagged = lagged_df[target_col]

    return lagged_df, X_lagged, y_lagged, selected_summary

In [88]:
lag_summary_df, lag_detail_dict = find_best_lags_for_all_features(
    df=selected_df,
    target_col=target_col,
    date_col="Date",
    max_lag=60,
    allow_negative_lag=False,
    min_obs=100
)

lagged_df, X_lagged, y_lagged, selected_lag_summary = build_lagged_feature_table(
    df=selected_df,
    summary_df=lag_summary_df,
    target_col=target_col,
    date_col="Date",
    min_abs_corr=0.03,
    max_features=100,
    keep_original_names=False
)

print("lagged_df shape:", lagged_df.shape)
print("X_lagged shape:", X_lagged.shape)
print("y_lagged shape:", y_lagged.shape)

lagged_df shape: (1472, 59)
X_lagged shape: (1472, 57)
y_lagged shape: (1472,)


In [89]:
lagged_df

,Date,target_5d_up,oil_volume_ma20_lag10,gold_volatility_5d_lag39,oil_volatility_5d_lag0,spy_close_ma60_ratio_lag0,gold_volume_ma20_ratio_lag55,gold_volume_change_20d_lag44,oil_volume_change_20d_lag27,gold_return_20d_lag50,...,spy_volume_change_20d_lag13,gold_return_1d_lag45,oil_volume_change_1d_lag28,qqq_volume_change_1d_lag21,qqq_volume_change_20d_lag13,open_close_return_lag52,volume_change_1d_lag23,oil_return_20d_lag0,gold_volume_change_1d_lag15,soxx_volume_change_1d_lag21
0,2020-06-19,0,11819615.0,0.010843,0.015470,0.073374,-0.425839,-0.296521,-0.577281,0.004677,...,-0.081683,-0.000865,-0.001041,0.032555,0.016516,0.050755,-0.433238,0.061278,0.289317,-0.119986
1,2020-06-22,0,12004085.0,0.011905,0.016352,0.077103,-0.544235,-0.467057,-0.632048,0.073753,...,0.163349,-0.019417,-0.030884,0.166392,-0.014673,-0.038371,-0.057583,0.133359,-0.353018,0.004906
2,2020-06-23,0,11989825.0,0.011487,0.018017,0.078338,-0.559703,-0.558691,-0.617400,0.126535,...,0.029359,0.007126,-0.005484,-0.340960,0.062019,0.015041,0.132441,0.119252,0.276246,-0.353650
3,2020-06-24,0,11898050.0,0.007281,0.031838,0.048200,-0.213001,-0.378017,-0.631443,0.148546,...,1.000318,-0.006825,0.144482,0.149254,0.226300,-0.033700,-0.020181,0.090000,0.636410,0.750487
4,2020-06-25,0,12533970.0,0.008437,0.032376,0.056359,-0.496079,-0.092889,-0.820259,0.127403,...,-0.036771,0.019671,-0.001274,0.684014,0.034063,0.017514,-0.439519,0.116260,-0.406253,0.228981
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1467,2026-04-23,1,55920465.0,0.016486,0.054884,0.045228,0.006160,-0.783380,9.834014,0.095138,...,-0.611610,0.022490,-0.362291,-0.357867,-0.593192,0.031989,0.013570,0.188112,-0.027604,-0.351307
1468,2026-04-24,0,52047335.0,0.015530,0.030535,0.052800,-0.162485,-0.256342,3.301106,0.109100,...,-0.318380,0.002793,0.119795,0.047178,-0.463316,0.019403,-0.008134,0.129115,-0.233105,0.109976
1469,2026-04-27,0,50182925.0,0.011668,0.028835,0.054017,-0.399866,-0.189965,5.891527,0.059750,...,0.148466,0.019715,-0.399914,0.347523,-0.015495,-0.008379,-0.515221,0.084702,-0.500185,0.733242
1470,2026-04-28,1,47576220.0,0.024384,0.023379,0.048332,-0.573914,-0.585809,7.171083,0.092812,...,-0.165211,0.027016,0.760579,0.014849,-0.370579,0.002271,0.196124,0.075252,0.340797,-0.377256


In [94]:
import gc
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, roc_auc_score


# =========================================================
# 1. running stats 업데이트
# =========================================================

def _update_running_stats(stats_dict, feature_name, values):
    if feature_name not in stats_dict:
        stats_dict[feature_name] = {
            "sum": 0.0,
            "sum_sq": 0.0,
            "count": 0,
        }

    vals = np.asarray(values, dtype=np.float64)

    stats_dict[feature_name]["sum"] += float(vals.sum())
    stats_dict[feature_name]["sum_sq"] += float((vals ** 2).sum())
    stats_dict[feature_name]["count"] += int(vals.size)


# =========================================================
# 2. 과적합 RF 기반 Permutation Importance
#    - 변수 선별용
#    - 최종 성능 검증용 아님
# =========================================================

def permutation_importance_rf_classifier_in_sample(
    lagged_df,
    target_col,
    date_col="Date",
    n_rf_runs=10,
    n_repeats=10,
    metric="logloss",
    random_state_base=42,
):
    """
    0/1 target용 과적합 RandomForestClassifier 기반 Permutation Importance.

    목적:
    - 최종 예측 성능 평가 X
    - 중요한 lagged 변수 후보 선정 O
    - 전체 데이터로 학습하고 전체 데이터에서 PI 계산

    metric="logloss":
        importance = permuted_logloss - baseline_logloss
        값이 클수록 중요한 변수

    metric="auc":
        importance = baseline_auc - permuted_auc
        단, 과적합 상태에서는 전부 0이 나올 수 있음
    """

    data = lagged_df.copy()
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna().reset_index(drop=True)

    if date_col in data.columns:
        data[date_col] = pd.to_datetime(data[date_col])
        data = data.sort_values(date_col).reset_index(drop=True)

    exclude_cols = [target_col]

    if date_col in data.columns:
        exclude_cols.append(date_col)

    X_df = data.drop(columns=exclude_cols)
    y = data[target_col].astype(int).to_numpy()

    feature_names = X_df.columns.tolist()

    if X_df.shape[0] == 0:
        raise ValueError("학습 데이터가 비어 있습니다.")

    if X_df.shape[1] == 0:
        raise ValueError("feature 컬럼이 없습니다.")

    if len(np.unique(y)) < 2:
        raise ValueError("target이 한 클래스만 존재합니다. 0/1 둘 다 있어야 합니다.")

    X = X_df.to_numpy(dtype=np.float64)

    stats_dict = {}

    baseline_rows = []

    for rf_run in range(n_rf_runs):
        rf = RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=None,
            class_weight="balanced",
            bootstrap=False,
            n_jobs=-1,
            random_state=random_state_base + rf_run,
        )

        rf.fit(X, y)

        baseline_proba = rf.predict_proba(X)[:, 1]

        baseline_logloss = log_loss(y, baseline_proba, labels=[0, 1])
        baseline_auc = roc_auc_score(y, baseline_proba)

        if metric == "logloss":
            baseline_score = baseline_logloss

        elif metric == "auc":
            baseline_score = baseline_auc

        else:
            raise ValueError("metric은 'logloss' 또는 'auc'만 가능합니다.")

        baseline_rows.append({
            "rf_run": rf_run + 1,
            "baseline_logloss": baseline_logloss,
            "baseline_auc": baseline_auc,
        })

        rng = np.random.RandomState(random_state_base + rf_run)

        for col_idx, feature_name in enumerate(feature_names):
            importances = np.empty(n_repeats, dtype=np.float64)

            for repeat_id in range(n_repeats):
                X_perm = X.copy()
                X_perm[:, col_idx] = rng.permutation(X_perm[:, col_idx])

                perm_proba = rf.predict_proba(X_perm)[:, 1]

                if metric == "logloss":
                    perm_score = log_loss(y, perm_proba, labels=[0, 1])
                    importances[repeat_id] = perm_score - baseline_score

                elif metric == "auc":
                    perm_score = roc_auc_score(y, perm_proba)
                    importances[repeat_id] = baseline_score - perm_score

                del X_perm, perm_proba

            _update_running_stats(
                stats_dict=stats_dict,
                feature_name=feature_name,
                values=importances,
            )

            del importances

        print(
            f"[PI] RF run {rf_run + 1}/{n_rf_runs} completed. "
            f"baseline_logloss={baseline_logloss:.6f}, "
            f"baseline_auc={baseline_auc:.4f}"
        )

        del rf, baseline_proba
        gc.collect()

    rows = []

    for feature_name, d in stats_dict.items():
        count = d["count"]

        mean_val = d["sum"] / count if count > 0 else np.nan

        if count > 1:
            var_val = (d["sum_sq"] - (d["sum"] ** 2) / count) / (count - 1)
            var_val = max(var_val, 0.0)
        else:
            var_val = 0.0

        rows.append({
            "feature": feature_name,
            "mean_importance": mean_val,
            "var_importance": var_val,
            "count": count,
        })

    importance_df = pd.DataFrame(rows)

    if importance_df.empty:
        return importance_df, pd.DataFrame(baseline_rows)

    importance_df["mean_importance"] = importance_df["mean_importance"].round(8)
    importance_df["var_importance"] = importance_df["var_importance"].round(8)

    # 중요도는 높고, 변동성은 낮은 변수 우선
    importance_df["score"] = (
        importance_df["mean_importance"] - importance_df["var_importance"]
    )

    importance_df["rank"] = (
        importance_df["score"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    importance_df = (
        importance_df
        .sort_values("rank")
        .reset_index(drop=True)
    )

    baseline_df = pd.DataFrame(baseline_rows)

    return importance_df, baseline_df

In [ ]:
importance_df, baseline_df = permutation_importance_rf_classifier_in_sample(
    lagged_df=lagged_df,
    target_col=target_col,
    date_col="Date",
    n_rf_runs=3,
    n_repeats=3,
    metric="logloss",
)

[PI] RF run 1/3 completed. baseline_logloss=0.000000, baseline_auc=1.0000
[PI] RF run 2/3 completed. baseline_logloss=0.000000, baseline_auc=1.0000
[PI] RF run 3/3 completed. baseline_logloss=0.000000, baseline_auc=1.0000


,feature,mean_importance,var_importance,count,score,rank
0,oil_volatility_5d_lag0,2.753246,0.066949,9,2.686297,1
1,spy_close_ma60_ratio_lag0,2.579994,0.003837,9,2.576157,2
2,gold_return_5d_lag21,1.212482,0.019940,9,1.192542,3
3,oil_volume_ma20_lag10,1.169319,0.011146,9,1.158173,4
4,oil_close_ma60_ratio_lag58,1.132060,0.009564,9,1.122497,5
5,oil_return_10d_lag58,0.873129,0.023906,9,0.849224,6
6,gold_close_ma60_ratio_lag42,0.791086,0.010136,9,0.780950,7
7,oil_return_3d_lag51,0.774466,0.011619,9,0.762847,8
8,gold_volume_change_5d_lag57,0.742939,0.015763,9,0.727176,9
9,soxx_return_5d_lag19,0.648893,0.006837,9,0.642056,10


In [96]:
importance_df

,feature,mean_importance,var_importance,count,score,rank
0,oil_volatility_5d_lag0,2.753246,6.694877e-02,9,2.686297,1
1,spy_close_ma60_ratio_lag0,2.579994,3.837040e-03,9,2.576157,2
2,gold_return_5d_lag21,1.212482,1.993994e-02,9,1.192542,3
3,oil_volume_ma20_lag10,1.169319,1.114647e-02,9,1.158173,4
4,oil_close_ma60_ratio_lag58,1.132060,9.563770e-03,9,1.122497,5
5,oil_return_10d_lag58,0.873129,2.390581e-02,9,0.849224,6
6,gold_close_ma60_ratio_lag42,0.791086,1.013554e-02,9,0.780950,7
7,oil_return_3d_lag51,0.774466,1.161864e-02,9,0.762847,8
8,gold_volume_change_5d_lag57,0.742939,1.576312e-02,9,0.727176,9
9,soxx_return_5d_lag19,0.648893,6.837090e-03,9,0.642056,10


In [97]:
top_k = 30

top_features = (
    importance_df
    .head(top_k)["feature"]
    .tolist()
)

top_model_df = lagged_df[["Date", target_col] + top_features].copy()

X_top = top_model_df[top_features].copy()
y_top = top_model_df[target_col].copy()

print("top_model_df shape:", top_model_df.shape)
print("X_top shape:", X_top.shape)
print("y_top shape:", y_top.shape)

top_model_df shape: (1472, 32)
X_top shape: (1472, 30)
y_top shape: (1472,)


In [98]:
top_model_df

,Date,target_5d_up,oil_volatility_5d_lag0,spy_close_ma60_ratio_lag0,gold_return_5d_lag21,oil_volume_ma20_lag10,oil_close_ma60_ratio_lag58,oil_return_10d_lag58,gold_close_ma60_ratio_lag42,oil_return_3d_lag51,...,gold_volume_ma5_ratio_lag17,soxx_volume_change_20d_lag7,gold_return_10d_lag58,soxx_volume_ma20_ratio_lag21,gold_volume_ma20_ratio_lag55,gold_return_3d_lag23,soxx_volatility_5d_lag4,spy_return_10d_lag50,gold_return_1d_lag45,spy_volume_change_1d_lag23
0,2020-06-19,0,0.015470,0.073374,0.019000,11819615.0,-0.544350,-0.357759,0.046737,-0.003914,...,0.125514,-0.314737,0.062605,-0.125325,-0.425839,0.006870,0.032438,0.110378,-0.000865,0.076993
1,2020-06-22,0,0.016352,0.077103,-0.004662,12004085.0,-0.562436,-0.300826,0.065834,-0.089831,...,-0.145878,-0.420553,0.079639,-0.129833,-0.544235,0.007668,0.034495,0.065084,-0.019417,-0.204793
2,2020-06-23,0,0.018017,0.078338,-0.004392,11989825.0,-0.557665,-0.262697,0.074593,-0.091241,...,0.086198,-0.292608,0.031276,-0.434826,-0.559703,0.004392,0.034738,0.087760,0.007126,-0.097990
3,2020-06-24,0,0.031838,0.048200,-0.011064,11898050.0,-0.532617,-0.070064,0.068327,-0.031434,...,-0.292353,-0.623058,0.062189,-0.022980,-0.213001,-0.002705,0.008061,0.084617,-0.006825,-0.088139
4,2020-06-25,0,0.032376,0.056359,-0.018751,12533970.0,-0.446830,-0.048417,0.059714,-0.132216,...,-0.036868,-0.315016,0.100406,0.184584,-0.496079,-0.006392,0.010218,0.077633,0.019671,-0.183101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1467,2026-04-23,1,0.054884,0.045228,-0.120060,55920465.0,0.117265,0.089932,0.133829,0.001798,...,-0.163153,-0.351151,0.164248,-0.355057,0.006160,-0.099919,0.008906,-0.004846,0.022490,0.470422
1468,2026-04-24,0,0.030535,0.052800,-0.063970,52047335.0,0.120842,0.117953,0.113981,0.017473,...,-0.040724,0.245545,0.051071,-0.289884,-0.162485,-0.091514,0.009751,-0.004975,0.002793,-0.176111
1469,2026-04-27,0,0.028835,0.054017,-0.060435,50182925.0,0.060936,0.051361,0.107277,0.024679,...,-0.047333,-0.308522,0.013862,0.254765,-0.399866,-0.052250,0.008729,-0.018399,0.019715,-0.284454
1470,2026-04-28,1,0.023379,0.048332,0.003193,47576220.0,0.089429,0.078068,0.112886,-0.021020,...,-0.217529,-0.309089,0.039018,-0.219010,-0.573914,0.007040,0.009989,-0.014769,0.027016,-0.060168


In [100]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)


# =========================================================
# 1. 시계열 train/test 분리
# =========================================================

def time_series_train_test_split(
    df,
    target_col,
    date_col="Date",
    test_size=0.2,
):
    data = df.copy()
    data = data.replace([np.inf, -np.inf], np.nan).dropna()

    if date_col in data.columns:
        data[date_col] = pd.to_datetime(data[date_col])
        data = data.sort_values(date_col).reset_index(drop=True)

    split_idx = int(len(data) * (1 - test_size))

    train_df = data.iloc[:split_idx].copy()
    test_df = data.iloc[split_idx:].copy()

    feature_cols = [
        col for col in data.columns
        if col not in [date_col, target_col]
    ]

    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].astype(int).copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].astype(int).copy()

    return train_df, test_df, X_train, X_test, y_train, y_test, feature_cols


# =========================================================
# 2. 여러 모델 학습/평가
# =========================================================

def train_final_models(
    top_model_df,
    target_col,
    date_col="Date",
    test_size=0.2,
):
    train_df, test_df, X_train, X_test, y_train, y_test, feature_cols = time_series_train_test_split(
        df=top_model_df,
        target_col=target_col,
        date_col=date_col,
        test_size=test_size,
    )

    models = {
        "logistic": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=500,
            max_depth=5,
            min_samples_split=10,
            min_samples_leaf=5,
            max_features="sqrt",
            class_weight="balanced",
            bootstrap=True,
            n_jobs=-1,
            random_state=42,
        ),
        "gradient_boosting": GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            random_state=42,
        ),
    }

    result_rows = []
    trained_models = {}

    for model_name, model in models.items():
        model.fit(X_train, y_train)

        pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_test)[:, 1]
        else:
            proba = pred.astype(float)

        if len(np.unique(y_test)) >= 2:
            auc = roc_auc_score(y_test, proba)
        else:
            auc = np.nan

        result_rows.append({
            "model": model_name,
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred, zero_division=0),
            "auc": auc,
            "train_rows": len(train_df),
            "test_rows": len(test_df),
            "n_features": len(feature_cols),
        })

        trained_models[model_name] = model

        print("=" * 70)
        print(f"[{model_name}]")
        print("confusion matrix:")
        print(confusion_matrix(y_test, pred))
        print()
        print(classification_report(y_test, pred, zero_division=0))

    result_df = pd.DataFrame(result_rows).sort_values(
        by=["auc", "f1"],
        ascending=False
    ).reset_index(drop=True)

    return result_df, trained_models, train_df, test_df, feature_cols


# =========================================================
# 3. 최종 모델 선택 후 전체 데이터 재학습
# =========================================================

def refit_best_model_on_full_data(
    top_model_df,
    target_col,
    best_model_name,
    trained_models,
    date_col="Date",
):
    data = top_model_df.copy()
    data = data.replace([np.inf, -np.inf], np.nan).dropna()

    if date_col in data.columns:
        data[date_col] = pd.to_datetime(data[date_col])
        data = data.sort_values(date_col).reset_index(drop=True)

    feature_cols = [
        col for col in data.columns
        if col not in [date_col, target_col]
    ]

    X = data[feature_cols].copy()
    y = data[target_col].astype(int).copy()

    best_model = trained_models[best_model_name]

    best_model.fit(X, y)

    return best_model, feature_cols, data


# =========================================================
# 4. 가장 최근 일자 기준 N_DAYS 뒤 상승 확률 예측
# =========================================================

def predict_latest_signal(
    fitted_model,
    fitted_data,
    feature_cols,
    target_col,
    date_col="Date",
    threshold=0.5,
):
    latest_row = fitted_data.iloc[[-1]].copy()

    latest_date = latest_row[date_col].iloc[0] if date_col in latest_row.columns else None

    X_latest = latest_row[feature_cols].copy()

    if hasattr(fitted_model, "predict_proba"):
        up_prob = fitted_model.predict_proba(X_latest)[:, 1][0]
    else:
        up_prob = fitted_model.predict(X_latest)[0]

    signal = 1 if up_prob >= threshold else 0

    result = {
        "latest_date": latest_date,
        "up_probability": up_prob,
        "threshold": threshold,
        "predicted_signal": signal,
    }

    return result

In [101]:
# =========================================================
# 최종 모델 학습/평가
# =========================================================

model_result_df, trained_models, train_df, test_df, final_feature_cols = train_final_models(
    top_model_df=top_model_df,
    target_col=target_col,
    date_col="Date",
    test_size=0.2,
)

model_result_df

best_model_name = model_result_df.iloc[0]["model"]

print("best_model_name:", best_model_name)

[logistic]
confusion matrix:
[[235   0]
 [ 60   0]]

              precision    recall  f1-score   support

           0       0.80      1.00      0.89       235
           1       0.00      0.00      0.00        60

    accuracy                           0.80       295
   macro avg       0.40      0.50      0.44       295
weighted avg       0.63      0.80      0.71       295

[random_forest]
confusion matrix:
[[211  24]
 [ 40  20]]

              precision    recall  f1-score   support

           0       0.84      0.90      0.87       235
           1       0.45      0.33      0.38        60

    accuracy                           0.78       295
   macro avg       0.65      0.62      0.63       295
weighted avg       0.76      0.78      0.77       295

[gradient_boosting]
confusion matrix:
[[225  10]
 [ 44  16]]

              precision    recall  f1-score   support

           0       0.84      0.96      0.89       235
           1       0.62      0.27      0.37        60

    accur

In [ ]:
final_model, final_feature_cols, fitted_data = refit_best_model_on_full_data(
    top_model_df=top_model_df,
    target_col=target_col,
    best_model_name=best_model_name,
    trained_models=trained_models,
    date_col="Date",
)

print("최종 사용 변수 개수:", len(final_feature_cols))

latest_signal = predict_latest_signal(
    fitted_model=final_model,
    fitted_data=fitted_data,
    feature_cols=final_feature_cols,
    target_col=target_col,
    date_col="Date",
    threshold=0.5,
)

latest_signal

최종 사용 변수 개수: 30


In [104]:
model_result_df[["model", "accuracy", "precision", "recall", "f1", "auc"]]

,model,accuracy,precision,recall,f1,auc
0,gradient_boosting,0.816949,0.615385,0.266667,0.372093,0.789574
1,random_forest,0.783051,0.454545,0.333333,0.384615,0.765461
2,logistic,0.796610,0.000000,0.000000,0.000000,0.455603
